In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
base = "/content/drive/MyDrive/HomeCredit/"
for f, expected in [("bureau.csv", 1716428),
                    ("previous_application.csv", 1670214),
                    ("installments_payments.csv", 13605401)]:
    rows = sum(1 for _ in open(base + f, encoding="utf-8")) - 1
    print(f, "Rows:", rows, "| Expected:", expected)

bureau.csv Rows: 1716428 | Expected: 1716428
previous_application.csv Rows: 1670214 | Expected: 1670214
installments_payments.csv Rows: 13605401 | Expected: 13605401


In [4]:
import pandas as pd, numpy as np, gc
base = "/content/drive/MyDrive/HomeCredit/"

app = pd.read_csv(base + "cleaned_data.csv")
print(app.shape)   # rows 307507

(307507, 85)


In [6]:
bureau = pd.read_csv(base + "bureau.csv", low_memory=False)
print("Bureau rows:", len(bureau))

for c in ['CREDIT_DAY_OVERDUE','AMT_CREDIT_SUM_DEBT','AMT_CREDIT_SUM',
          'AMT_CREDIT_MAX_OVERDUE','DAYS_CREDIT']:
    bureau[c] = pd.to_numeric(bureau[c], errors='coerce')

bureau['IS_ACTIVE'] = (bureau['CREDIT_ACTIVE'] == 'Active').astype(int)
bureau['IS_OVERDUE'] = (bureau['CREDIT_DAY_OVERDUE'] > 0).astype(int)

bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    BUREAU_LOAN_COUNT=('SK_ID_BUREAU', 'count'),
    BUREAU_ACTIVE_COUNT=('IS_ACTIVE', 'sum'),
    BUREAU_ACTIVE_RATIO=('IS_ACTIVE', 'mean'),
    BUREAU_OVERDUE_COUNT=('IS_OVERDUE', 'sum'),
    BUREAU_TOTAL_DEBT=('AMT_CREDIT_SUM_DEBT', 'sum'),
    BUREAU_TOTAL_CREDIT=('AMT_CREDIT_SUM', 'sum'),
    BUREAU_MAX_OVERDUE=('AMT_CREDIT_MAX_OVERDUE', 'max'),
    BUREAU_AVG_DAYS_CREDIT=('DAYS_CREDIT', 'mean'),
).reset_index()

bureau_agg['BUREAU_DEBT_CREDIT_RATIO'] = (
    bureau_agg['BUREAU_TOTAL_DEBT'] / bureau_agg['BUREAU_TOTAL_CREDIT'].replace(0, np.nan))

print(bureau_agg.shape)
del bureau; gc.collect()

Bureau rows: 1716428
(305811, 10)


7

In [7]:
prev = pd.read_csv(base + "previous_application.csv", low_memory=False)
print("Prev rows:", len(prev))

for c in ['AMT_CREDIT','AMT_ANNUITY','AMT_DOWN_PAYMENT']:
    prev[c] = pd.to_numeric(prev[c], errors='coerce')

prev['IS_APPROVED'] = (prev['NAME_CONTRACT_STATUS'] == 'Approved').astype(int)
prev['IS_REFUSED'] = (prev['NAME_CONTRACT_STATUS'] == 'Refused').astype(int)

prev_agg = prev.groupby('SK_ID_CURR').agg(
    PREV_APP_COUNT=('SK_ID_PREV', 'count'),
    PREV_APPROVED_COUNT=('IS_APPROVED', 'sum'),
    PREV_REFUSED_COUNT=('IS_REFUSED', 'sum'),
    PREV_REFUSED_RATIO=('IS_REFUSED', 'mean'),
    PREV_AVG_CREDIT=('AMT_CREDIT', 'mean'),
    PREV_AVG_ANNUITY=('AMT_ANNUITY', 'mean'),
    PREV_AVG_DOWN_PAYMENT=('AMT_DOWN_PAYMENT', 'mean'),
).reset_index()

print(prev_agg.shape)
del prev; gc.collect()

Prev rows: 1670214
(338857, 8)


0

In [8]:
use_cols = ['SK_ID_PREV','SK_ID_CURR','DAYS_INSTALMENT',
            'DAYS_ENTRY_PAYMENT','AMT_INSTALMENT','AMT_PAYMENT']
inst = pd.read_csv(base + "installments_payments.csv", usecols=use_cols, low_memory=False)
print("Installments rows:", len(inst))

for c in ['DAYS_INSTALMENT','DAYS_ENTRY_PAYMENT','AMT_INSTALMENT','AMT_PAYMENT']:
    inst[c] = pd.to_numeric(inst[c], errors='coerce').astype('float32')

inst['DAYS_LATE'] = inst['DAYS_ENTRY_PAYMENT'] - inst['DAYS_INSTALMENT']
inst['IS_LATE'] = (inst['DAYS_LATE'] > 0).astype('int8')
inst['PAYMENT_DIFF'] = inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']

inst_agg = inst.groupby('SK_ID_CURR').agg(
    INST_COUNT=('SK_ID_PREV', 'count'),
    INST_LATE_RATIO=('IS_LATE', 'mean'),
    INST_AVG_DAYS_LATE=('DAYS_LATE', 'mean'),
    INST_MAX_DAYS_LATE=('DAYS_LATE', 'max'),
    INST_AVG_PAYMENT_DIFF=('PAYMENT_DIFF', 'mean'),
).reset_index()

print(inst_agg.shape)
del inst; gc.collect()

Installments rows: 13605401
(339587, 6)


17

In [9]:
features = (app
            .merge(bureau_agg, on='SK_ID_CURR', how='left')
            .merge(prev_agg, on='SK_ID_CURR', how='left')
            .merge(inst_agg, on='SK_ID_CURR', how='left'))

count_cols = ['BUREAU_LOAN_COUNT','BUREAU_ACTIVE_COUNT','BUREAU_OVERDUE_COUNT',
              'PREV_APP_COUNT','PREV_APPROVED_COUNT','PREV_REFUSED_COUNT','INST_COUNT']
features[count_cols] = features[count_cols].fillna(0)

print(features.shape)
print(features['TARGET'].mean())

(307507, 106)
0.08072986956394489


In [10]:
new_cols = [c for c in features.columns if c.startswith(('BUREAU_','PREV_','INST_'))]
features[new_cols + ['TARGET']].corr()['TARGET'].drop('TARGET').sort_values()

,TARGET
PREV_AVG_ANNUITY,-0.034870
PREV_AVG_DOWN_PAYMENT,-0.024624
PREV_APPROVED_COUNT,-0.023517
INST_COUNT,-0.016308
PREV_AVG_CREDIT,-0.016112
BUREAU_TOTAL_CREDIT,-0.014057
BUREAU_LOAN_COUNT,-0.010017
BUREAU_MAX_OVERDUE,0.002540
INST_MAX_DAYS_LATE,0.004657
BUREAU_TOTAL_DEBT,0.007145


In [11]:
from google.colab import files
features.to_csv("features.csv", index=False)
files.download("features.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>